In [0]:
# Service Principal credentials
application_id = "********************"
authentication_key = "***************************"
tenant_id = "************************"

# Set Spark config for ADLS access
spark.conf.set("fs.azure.account.auth.type.delearn.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.delearn.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.delearn.dfs.core.windows.net", application_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.delearn.dfs.core.windows.net", authentication_key)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.delearn.dfs.core.windows.net", "https://login.microsoftonline.com/" + tenant_id + "/oauth2/token")

print("✅ Spark config set!")

✅ Spark config set!


In [0]:
from pyspark.sql.functions import lit
from pyspark.sql.types import TimestampType

# =========================
# 1. PATHS
# =========================
full_load_path = "abfss://cdcbronze@delearn.dfs.core.windows.net/full_load/"
silver_path = "abfss://cdcsilver@delearn.dfs.core.windows.net/customer/"

# =========================
# 2. READ FULL LOAD DATA
# =========================
df_full = spark.read.parquet(full_load_path)

print("Full Load Data:")
display(df_full)

# =========================
# 3. ADD SOFT DELETE COLUMNS
# =========================
df_full_soft = (
    df_full
    .withColumn("IsDeleted", lit(False))
    .withColumn("DeletedAt", lit(None).cast(TimestampType()))
)

print("Full Load Data with Soft Delete Columns:")
display(df_full_soft)

# =========================
# 4. WRITE TO SILVER AS DELTA
# =========================
df_full_soft.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("Soft Delete Silver Delta created successfully at:", silver_path)


# =========================
# 5. VERIFY
# =========================
print("Final Silver Table:")
display(spark.read.format("delta").load(silver_path))

Full Load Data:


CustomerID,CustomerName,City,Email,UpdatedAt
1,Ravi,Hyderabad,ravi@gmail.com,2026-06-12T11:12:39.474409Z
2,Sita,Pune,sita@gmail.com,2026-06-12T11:12:39.474409Z
3,John,Delhi,john@gmail.com,2026-06-12T11:12:39.474409Z


Full Load Data with Soft Delete Columns:


CustomerID,CustomerName,City,Email,UpdatedAt,IsDeleted,DeletedAt
1,Ravi,Hyderabad,ravi@gmail.com,2026-06-12T11:12:39.474409Z,false,null
2,Sita,Pune,sita@gmail.com,2026-06-12T11:12:39.474409Z,false,null
3,John,Delhi,john@gmail.com,2026-06-12T11:12:39.474409Z,false,null


Soft Delete Silver Delta created successfully at: abfss://cdcsilver@delearn.dfs.core.windows.net/customer/


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8242456114718282>, line 42
     37 print("Soft Delete Silver Delta created successfully at:", silver_path)
     39 # =========================
     40 # 5. REGISTER TABLE (OPTIONAL)
     41 # =========================
---> 42 spark.sql(f"""
     43 CREATE TABLE IF NOT EXISTS silver_customer
     44 USING DELTA
     45 LOCATION '{silver_path}'
     46 """)
     48 print("silver_customer table registered successfully.")
     50 # =========================
     51 # 6. VERIFY
     52 # =========================

File /databricks/spark/python/pyspark/databricks/instrumentation/instrumentation_utils.py:217, in _wrap_function.<locals>.wrapper(*args, **kwargs)
    215 start = time.perf_counter()
    216 try:
--> 217     res = func(*args, **kwargs)
    218     logging_helper.log_event(
    219         accessor=wrapper,
    220    

In [0]:
df_check = spark.read.parquet(full_load_path)
display(df_check)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8079855002243519>, line 1
----> 1 df_check = spark.read.parquet(full_load_path)
      2 display(df_check)

NameError: name 'full_load_path' is not defined